In [1]:
from ase.io import read, write
import os
import shutil

def write_poscar_with_selective_dynamics(filename, atoms, selective_dynamics):
    """
    Write a POSCAR file with Selective Dynamics flags.
    
    Args:
        filename (str): Name of the output POSCAR file.
        atoms (ase.Atoms): The atomic structure.
        selective_dynamics (list): List of [X, Y, Z] flags for each atom.
    """
    with open(filename, 'w') as f:
        # Write the header
        f.write('System Name\n')
        f.write('1.10343677\n')
        
        # Write the cell vectors
        for vector in atoms.cell:
            f.write(f'{vector[0]:20.16f} {vector[1]:20.16f} {vector[2]:20.16f}\n')
        
        # Write the atomic symbols and counts
        symbols = atoms.get_chemical_symbols()
        unique_symbols = sorted(set(symbols), key=symbols.index)
        counts = [symbols.count(symbol) for symbol in unique_symbols]
        f.write(' '.join(unique_symbols) + '\n')
        f.write(' '.join(map(str, counts)) + '\n')
        
        # Write Selective Dynamics line
        f.write('Selective dynamics\n')
        
        # Write Cartesian or Direct coordinates
        f.write('Cartesian\n')
        
        # Write atomic positions and Selective Dynamics flags
        for i, atom in enumerate(atoms):
            pos = atom.position
            flags = selective_dynamics[i]
            f.write(f'{pos[0]:20.16f} {pos[1]:20.16f} {pos[2]:20.16f} ')
            f.write(f'{"T" if flags[0] else "F"} {"T" if flags[1] else "F"} {"T" if flags[2] else "F"}\n')


# SLURM job script template
slurm_template = """#!/bin/bash
#SBATCH --job-name=Nb_{i}_{j}
#SBATCH --output=slurm_{i}_{j}.out
#SBATCH --error=slurm_{i}_{j}.err
#SBATCH --nodes=2
#SBATCH --ntasks-per-node=15
#SBATCH --cpus-per-task=1
#SBATCH --time=8:00:00
#SBATCH --partition=parallelshort
#SBATCH --mem-per-cpu=8G

# Load necessary modules (if required)
module load VASP

# Run the calculation
srun vasp_std
"""

# List of source files
source_files = [
    '/scratch/p301616/VASP_DFT_bcc_DB/niobium_NewGammaSurf_110/INPUTS_temp/INCAR',
    '/scratch/p301616/VASP_DFT_bcc_DB/niobium_NewGammaSurf_110/INPUTS_temp/POTCAR'
]

###########################M
# MAIN CODE
############################

# Read the XYZ file

ny = 10
nz = 24
perf_110 = read('110_plane.xyz')

# Loop through the first 1000 items in primitive_xyz
for i in range(ny):
    for j in range(nz):
        direc = 'Nb_gammaSurf_' + str(i) + '_' + str(j)

        os.makedirs(direc, exist_ok=True)
        os.chdir(direc)
        primitive_xyz = perf_110.copy()
        # Define Selective Dynamics flags (F T T for all atoms)
        selective_dynamics = [[True, False, False] for _ in range(len(primitive_xyz))]
        # add the displacement
        primitive_xyz.cell[0][1] += i * primitive_xyz.cell[1][1]/ny
        primitive_xyz.cell[0][2] += j * primitive_xyz.cell[2][2]/nz

        # Write the POSCAR file with Selective Dynamics
        write_poscar_with_selective_dynamics('POSCAR', primitive_xyz, selective_dynamics)
        
        # Get the current directory
        current_directory = os.getcwd()
        # copy INCAR and POTCAR files
        for source_file in source_files:
            shutil.copy(source_file, current_directory)

        # Write the SLURM job script
        with open('job_{}_{}.sh'.format(i,j), 'w') as f:
            f.write(slurm_template.format(i=i,j=j))

        # Submit the SLURM job
        os.system('sbatch job_{}_{}.sh'.format(i,j))

        print('job {} + {} submitted!'.format(i,j))
        os.chdir("../")

Submitted batch job 17961826
job 0 + 0 submitted!
job 0 + 1 submitted!
Submitted batch job 17961827
Submitted batch job 17961828
job 0 + 2 submitted!
Submitted batch job 17961829
job 0 + 3 submitted!
Submitted batch job 17961830
job 0 + 4 submitted!
Submitted batch job 17961831
job 0 + 5 submitted!
Submitted batch job 17961832
job 0 + 6 submitted!
Submitted batch job 17961833
job 0 + 7 submitted!
Submitted batch job 17961834
job 0 + 8 submitted!
Submitted batch job 17961835
job 0 + 9 submitted!
Submitted batch job 17961836
job 0 + 10 submitted!
Submitted batch job 17961837
job 0 + 11 submitted!
Submitted batch job 17961838
job 0 + 12 submitted!
Submitted batch job 17961839
job 0 + 13 submitted!
Submitted batch job 17961840
job 0 + 14 submitted!
Submitted batch job 17961841
job 0 + 15 submitted!
Submitted batch job 17961842
job 0 + 16 submitted!
Submitted batch job 17961843
job 0 + 17 submitted!
Submitted batch job 17961844
job 0 + 18 submitted!
Submitted batch job 17961845
job 0 + 19 s